In [ ]:
EPOCHS, PATIENCE = 50, 7         
EXPERIMENT = "cipherstop-lstm-attention"

In [ ]:
import os, sys, glob, shutil, subprocess, json
import pandas as pd

def find_dir(name, base="/kaggle/input", maxdepth=7):
    for root, dirs, _ in os.walk(base):
        if root[len(base):].count(os.sep) > maxdepth:
            dirs[:] = []; continue
        if name in dirs:
            return os.path.join(root, name)
    return None

def find_file(name, base="/kaggle/input", maxdepth=7):
    for root, dirs, files in os.walk(base):
        if root[len(base):].count(os.sep) > maxdepth:
            dirs[:] = []; continue
        if name in files:
            return os.path.join(root, name)
    return None

PROC = find_dir("processed_split")
MANIFEST = find_file("manifest.csv")
print("processed_split:", PROC)
print("manifest.csv    :", MANIFEST)
assert PROC and MANIFEST, "Add the build notebook output as an input (see the top of this notebook)."

WORK = "/kaggle/working"
ROOT = f"{WORK}/CipherStop"
CKPT, OUT, MLRUNS = f"{WORK}/checkpoints", f"{WORK}/zeroday_results", f"{WORK}/mlruns"
os.makedirs(f"{ROOT}/src/model", exist_ok=True)
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"      
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
TRACKING = f"file://{MLRUNS}"

processed_split: /kaggle/input/notebooks/ayettmansouri/01-ingestionv2/processed_split
manifest.csv    : /kaggle/input/notebooks/ayettmansouri/01-ingestionv2/manifest.csv


In [ ]:
man = pd.read_csv(MANIFEST)
print("manifest splits:", man.split.value_counts().to_dict())
for cut in (10, 25, 50, 75):
    d = pd.read_parquet(f"{PROC}/{cut}pct", columns=["rel_dir", "family", "label", "split"])
    ryuk = d[d.family == "Ryuk"].split.value_counts().to_dict()
    print(f"{cut}%: runs per split = {d.split.value_counts().to_dict()} | Ryuk per split = {ryuk}")
    assert "split" in d.columns
    assert set(ryuk) <= {"zeroday"}, "Ryuk must ONLY be in the zeroday split"
    for sp in ("train", "val", "test"):
        assert (d.split == sp).sum() == (man.split == sp).sum(), f"{cut}%: {sp} run count differs from the manifest"
print("OK: official split, Ryuk is never in train/val/test")

manifest splits: {'train': 770, 'test': 330, 'extra': 330, 'val': 220, 'zeroday': 150, 'mix': 100, 'variants': 70}
10%: runs per split = {'train': 770, 'test': 330, 'val': 220, 'zeroday': 150} | Ryuk per split = {'zeroday': 150}
25%: runs per split = {'train': 770, 'test': 330, 'val': 220, 'zeroday': 150} | Ryuk per split = {'zeroday': 150}
50%: runs per split = {'train': 770, 'test': 330, 'val': 220, 'zeroday': 150} | Ryuk per split = {'zeroday': 150}
75%: runs per split = {'train': 770, 'test': 330, 'val': 220, 'zeroday': 150} | Ryuk per split = {'zeroday': 150}
OK: official split, Ryuk is never in train/val/test


In [ ]:
# code: src/model/lstm.py 
open(f'{ROOT}/src/__init__.py', 'w').close()
open(f'{ROOT}/src/model/__init__.py', 'w').close()
open(f'{ROOT}/src/model/lstm.py', 'w').write(r'''"""CipherStop - LSTM + Attention model for early ransomware detection.

Input : a batch of behavioral sequences, shape (batch, seq_len, n_features),
        plus the true length of each sequence (the rest is padding).
Output: a detection score in [0, 1] per sequence (1 = ransomware).
"""
from typing import Optional, Tuple, Union

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class Attention(nn.Module):
    """Additive attention over the LSTM outputs.

    For each time step t:  e_t = v^T tanh(W h_t)
    Weights: alpha = softmax(e) over the *real* (non-padded) steps only.
    Context vector: sum_t alpha_t * h_t
    """

    def __init__(self, hidden_dim: int, attn_dim: int = 64):
        super().__init__()
        self.proj = nn.Linear(hidden_dim, attn_dim)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(
        self, h: torch.Tensor, mask: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # h: (B, T, H), mask: (B, T) True for real steps
        scores = self.v(torch.tanh(self.proj(h))).squeeze(-1)  # (B, T)
        scores = scores.masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)  # (B, T), padding -> 0
        context = torch.bmm(weights.unsqueeze(1), h).squeeze(1)  # (B, H)
        return context, weights


class RansomwareLSTM(nn.Module):
    """LSTM + attention binary classifier for variable-length sequences.

    The LSTM is unidirectional on purpose: for early detection the model
    must only use the past, exactly like in real-time monitoring.
    """

    def __init__(
        self,
        n_features: int,
        hidden_dim: int = 128,
        num_layers: int = 2,
        dropout: float = 0.3,
        attn_dim: int = 64,
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attention = Attention(hidden_dim, attn_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, 1)

    def logits(
        self, x: torch.Tensor, lengths: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Raw scores before sigmoid (use these with BCEWithLogitsLoss)."""
        B, T, _ = x.shape
        if lengths is None:
            lengths = torch.full((B,), T, dtype=torch.long)
        lengths = lengths.to("cpu").clamp(min=1, max=T)

        packed = pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=T)

        mask = torch.arange(T, device=x.device).unsqueeze(0) < lengths.to(
            x.device
        ).unsqueeze(1)
        context, weights = self.attention(out, mask)
        logit = self.classifier(self.dropout(context)).squeeze(-1)  # (B,)
        return logit, weights

    def forward(
        self,
        x: torch.Tensor,
        lengths: Optional[torch.Tensor] = None,
        return_attention: bool = False,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        """Returns detection score in [0, 1], shape (B,)."""
        logit, weights = self.logits(x, lengths)
        score = torch.sigmoid(logit)
        return (score, weights) if return_attention else score
''')
print('lstm.py written')

lstm.py written


In [ ]:
# code: train.py 
open(f'{ROOT}/train.py', 'w').write(r'''"""CipherStop - training loop with MLflow logging (KAN-27/28/29/30).

Expected data layout (output of the Spark sequence-generation notebook):
    <data-root>/10pct/  25pct/  50pct/  75pct/      (parquet folders)
Each parquet has at least: sequence (array<array<double>>), label (0/1).
If a `split` column (train / val / test) exists it is used; otherwise a
stratified 70/15/15 split is made (with a warning).
"""
import argparse
import os
import random
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (ConfusionMatrixDisplay, confusion_matrix, f1_score,roc_auc_score)
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

from src.model.lstm import RansomwareLSTM


class SeqDataset(Dataset):
    def __init__(self, seqs, labels):
        self.seqs, self.labels = seqs, labels

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, i):
        return self.seqs[i], self.labels[i]


def collate(batch):
    """Pad variable-length sequences and return the real lengths."""
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    x = pad_sequence(seqs, batch_first=True)  # zero padding
    return x, lengths, torch.tensor(labels, dtype=torch.float32)


def to_array(seq):
    return np.stack([np.asarray(step, dtype=np.float32) for step in seq])


def load_cut(path: Path):
    df = pd.read_parquet(path)
    df = df[df["sequence"].map(len) > 0].reset_index(drop=True)
    if "split" in df.columns:
        parts = {s: df[df["split"] == s] for s in ("train", "val", "test")}
    else:
        print("WARNING: no `split` column - using a random stratified split. "
              "Prefer the official split to avoid leakage between executions.")
        tr, rest = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=42)
        va, te = train_test_split(rest, test_size=0.5, stratify=rest["label"], random_state=42)
        parts = {"train": tr, "val": va, "test": te}
    out = {}
    for name, p in parts.items():
        out[name] = ([to_array(s) for s in p["sequence"]], p["label"].astype(int).tolist())
    return out


def preprocess(parts):
    """log1p (counts/sizes are heavy-tailed) + standardise with TRAIN stats."""
    def log(seqs):
        return [np.log1p(np.clip(s, 0, None)) for s in seqs]

    parts = {k: (log(v[0]), v[1]) for k, v in parts.items()}
    allx = np.concatenate(parts["train"][0], axis=0)
    mean, std = allx.mean(0), allx.std(0) + 1e-6
    loaders_data = {}
    for k, (seqs, labels) in parts.items():
        tens = [torch.from_numpy((s - mean) / std) for s in seqs]
        loaders_data[k] = SeqDataset(tens, labels)
    return loaders_data, mean, std


#  metrics
def compute_metrics(y_true, y_prob, thr=0.5):
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    auc = roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float("nan")
    return {
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "fpr": fp / (fp + tn) if (fp + tn) else 0.0,
        "auc": auc,
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    losses, probs, labels = [], [], []
    for x, lengths, y in loader:
        x, y = x.to(device), y.to(device)
        logit, _ = model.logits(x, lengths)
        losses.append(criterion(logit, y).item() * len(y))
        probs += torch.sigmoid(logit).cpu().tolist()
        labels += y.cpu().tolist()
    return sum(losses) / len(labels), np.array(labels), np.array(probs)


class EarlyStopping:
    """Stop when the monitored metric (higher = better) stops improving."""

    def __init__(self, patience):
        self.patience, self.best, self.bad = patience, -np.inf, 0

    def step(self, value):
        if value > self.best:
            self.best, self.bad = value, 0
            return True  
        self.bad += 1
        return False

    @property
    def stop(self):
        return self.bad >= self.patience


def log_confusion_matrix(y_true, y_prob, name, out_dir):
    y_pred = (y_prob >= 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay(cm, display_labels=["benign", "ransomware"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
    path = out_dir / f"confusion_matrix_{name}.png"
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)
    mlflow.log_artifact(str(path), artifact_path="confusion_matrix")


#  training
def train_one_cut(cut, args, device):
    data_path = Path(args.data_root) / f"{cut}pct"
    parts = load_cut(data_path)
    datasets, mean, std = preprocess(parts)
    n_features = datasets["train"][0][0].shape[1]

    mk = lambda ds, shuffle: DataLoader(ds, batch_size=args.batch_size, shuffle=shuffle,
                                        collate_fn=collate)
    train_dl, val_dl, test_dl = mk(datasets["train"], True), mk(datasets["val"], False), mk(datasets["test"], False)

    n_pos = sum(datasets["train"].labels)
    n_neg = len(datasets["train"]) - n_pos
    lens = [len(s) for s in datasets["train"].seqs]
    print(f"[{cut}%] sizes train/val/test = {len(datasets['train'])}/{len(datasets['val'])}/"
          f"{len(datasets['test'])} | train pos/neg = {n_pos}/{n_neg} | "
          f"seq_len min/mean/max = {min(lens)}/{np.mean(lens):.1f}/{max(lens)}")
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    model = RansomwareLSTM(n_features, args.hidden_dim, args.num_layers, args.dropout).to(device)
    optim = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

    ckpt_dir = Path(args.ckpt_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    with mlflow.start_run(run_name=f"lstm_attn_{cut}pct") as run:
        mlflow.set_tag("sequence_cut", f"{cut}pct")
        mlflow.log_params({
            **{k: v for k, v in vars(args).items() if k not in ("cuts",)},
            "cut": cut, "n_features": n_features, "n_train": len(datasets["train"]),
            "n_val": len(datasets["val"]), "n_test": len(datasets["test"]),
            "pos_weight": round(pos_weight.item(), 3),
        })
        ckpt_path = ckpt_dir / f"lstm_attn_{cut}pct_{run.info.run_id[:8]}.pt"
        stopper = EarlyStopping(args.patience)

        for epoch in range(1, args.epochs + 1):
            model.train()
            tot, n = 0.0, 0
            for x, lengths, y in train_dl:
                x, y = x.to(device), y.to(device)
                optim.zero_grad()
                logit, _ = model.logits(x, lengths)
                loss = criterion(logit, y)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optim.step()
                tot, n = tot + loss.item() * len(y), n + len(y)

            _, ytr, ptr = evaluate(model, train_dl, criterion, device)
            val_loss, yva, pva = evaluate(model, val_dl, criterion, device)
            tm, vm = compute_metrics(ytr, ptr), compute_metrics(yva, pva)
            mlflow.log_metrics({
                "train_loss": tot / n, "val_loss": val_loss,
                "train_f1": tm["f1"], "train_fpr": tm["fpr"], "train_auc": tm["auc"],
                "val_f1": vm["f1"], "val_fpr": vm["fpr"], "val_auc": vm["auc"],
            }, step=epoch)
            print(f"[{cut}%] epoch {epoch:03d} loss={tot/n:.4f} val_loss={val_loss:.4f} "
                  f"val_auc={vm['auc']:.4f} val_f1={vm['f1']:.4f} val_fpr={vm['fpr']:.4f}")

            monitor = vm["auc"] if not np.isnan(vm["auc"]) else -val_loss
            if stopper.step(monitor):
                torch.save({"state_dict": model.state_dict(), "n_features": n_features,
                            "scaler_mean": mean, "scaler_std": std, "args": vars(args),
                            "epoch": epoch, "val_auc": vm["auc"]}, ckpt_path)
            if stopper.stop:
                print(f"Early stopping at epoch {epoch}")
                break

        #  reload the best model, evaluate on test, log artifacts
        best = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(best["state_dict"])
        mlflow.log_metric("best_epoch", best["epoch"])
        for split_name, dl in (("val", val_dl), ("test", test_dl)):
            loss, y, p = evaluate(model, dl, criterion, device)
            m = compute_metrics(y, p)
            mlflow.log_metrics({f"final_{split_name}_{k}": v for k, v in m.items()})
            log_confusion_matrix(y, p, f"{cut}pct_{split_name}", ckpt_dir) 
        mlflow.log_artifact(str(ckpt_path), artifact_path="checkpoints")     
        try: 
        except TypeError:   
            mlflow.pytorch.log_model(model, artifact_path="model")
        except Exception as e:
            print("WARNING: mlflow.pytorch.log_model skipped:", str(e)[:200])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", default="data/processed")
    ap.add_argument("--cuts", type=int, nargs="+", default=[10, 25, 50, 75])
    ap.add_argument("--tracking-uri", default=os.getenv("MLFLOW_TRACKING_URI"))
    ap.add_argument("--experiment", default="cipherstop-lstm-attention")
    ap.add_argument("--ckpt-dir", default="checkpoints")
    ap.add_argument("--epochs", type=int, default=50)
    ap.add_argument("--patience", type=int, default=7)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--weight-decay", type=float, default=1e-5)
    ap.add_argument("--hidden-dim", type=int, default=128)
    ap.add_argument("--num-layers", type=int, default=2)
    ap.add_argument("--dropout", type=float, default=0.3)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if args.tracking_uri:
        mlflow.set_tracking_uri(args.tracking_uri)
    mlflow.set_experiment(args.experiment)
    print("Tracking URI:", mlflow.get_tracking_uri(), "| device:", device)

    for cut in args.cuts:
        train_one_cut(cut, args, device)


if __name__ == "__main__":
    main()
''')
print('train.py written')

train.py written


In [ ]:
#  code: zeroday_eval_torch.py
open(f'{ROOT}/zeroday_eval_torch.py', 'w').write(r'''"""RQ3 - zero-day evaluation (unseen family = Ryuk) for the CipherStop PyTorch LSTM+Attention.

Frozen checkpoints produced by train.py (lstm_attn_{cut}pct_{runid8}.pt) are scored on
  zeroday     : every Ryuk run + held-out benign runs (test split)
  zeroday_hw  : Ryuk on TEST hardware only + the same benign runs (hardware-matched)
  seen_test   : seen families, held-out hardware   (reference -> the seen-vs-unseen gap)
  variants    : 7 Conti variants (recall only), if present

Preprocessing is read from the checkpoint (scaler_mean / scaler_std / n_features / args),
and reproduces train.py exactly: log1p(clip(x,0)) -> (x-mean)/std -> pack by true length.
No threshold, scaler or hyper-parameter is ever fitted on Ryuk.

Run from the repo root (so `src.model.lstm` imports). Windows / local default layout:
  data/processed/{10,25,50,75}pct   checkpoints/lstm_attn_{cut}pct_{runid8}.pt   data/manifest.csv
  python zeroday_eval_torch.py --inspect                 # 1) look at the data first
  python zeroday_eval_torch.py                           # 2) run the evaluation
  (add --tracking-uri http://localhost:5000 to verify split sizes and log to MLflow)

Data: data/processed/{cut}pct parquets (rel_dir,label,sequence[,split]) must cover the FULL manifest
(incl. Ryuk, val, test). --merged-root is only an alternative input (merged_{split} window parquets).
"""
import argparse
import glob
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from torch.nn.utils.rnn import pad_sequence

FRACTIONS = [10, 25, 50, 75]
ZERODAY_FAMILY = "Ryuk"
TEST_HW = {"celeron-gen6/ddr4-2133-8g", "i3-gen12/ddr4-3200-16g", "i3-gen12/ddr4-2666-8g"}
META_COLS = ["rel_dir", "family", "top", "cpu", "ram", "hw", "label"]


# data
def seqs_from_merged(root, split, fraction, fill_gaps):
    """Same logic as the sequence-generation notebook: sort by step, features in column order,
    keep the first max(1, int(n*fraction)) windows."""
    p = Path(root) / f"merged_{split}"
    if not p.exists():
        return None
    d = pd.read_parquet(p)
    feat = [c for c in d.columns if c not in META_COLS + ["step"]]
    rows = []
    for rid, g in d.groupby("rel_dir"):
        g = g.sort_values("step").set_index("step")[feat]
        if fill_gaps:
            g = g.reindex(range(int(g.index.max()) + 1), fill_value=0)
        x = g.to_numpy(dtype=np.float32)
        rows.append(dict(rel_dir=rid, family=group_first(d, rid, "family"),
                         label=int(group_first(d, rid, "label")),
                         sequence=list(x[:max(1, int(len(x) * fraction))])))
    return pd.DataFrame(rows)


def group_first(d, rid, col):
    return d.loc[d.rel_dir == rid, col].iloc[0]


def load_fraction(args, manifest, frac):
    if args.merged_root:
        parts = []
        for sp in ["train", "val", "test", "zeroday", "variants"]:
            r = seqs_from_merged(args.merged_root, sp, frac / 100, args.fill_gaps)
            if r is not None:
                parts.append(r)
        df = pd.concat(parts, ignore_index=True)
    else:
        df = pd.read_parquet(Path(args.processed_root) / f"{frac}pct")
    df = df[df["sequence"].map(len) > 0]
    cols = ["rel_dir", "split", "hw", "top"]
    n0 = len(df)
    df = df.drop(columns=[c for c in cols[1:] if c in df.columns]).merge(manifest[cols], on="rel_dir", how="inner")
    if len(df) == 0:
        raise SystemExit(f"{frac}%: no rel_dir of the parquet matches the manifest (run --inspect)")
    if len(df) < n0:
        print(f"[{frac}%] {n0 - len(df)} parquet runs not in manifest were dropped")
    if not (df.family == ZERODAY_FAMILY).any():
        raise SystemExit(f"{frac}%: no Ryuk runs in data/processed -> regenerate the sequences for the zeroday split")
    if not (df.split == "test").any():
        raise SystemExit(f"{frac}%: no test-split runs in the parquet")
    df["seq"] = df["sequence"].map(lambda s: np.stack([np.asarray(r, dtype=np.float32) for r in s]))
    return df.drop(columns=["sequence"]).reset_index(drop=True)


def find_repo_root():
    """Folder that contains src/model/lstm.py, searched upward from the cwd and from this file."""
    for start in (Path.cwd(), Path(__file__).resolve().parent):
        for p in [start, *start.parents]:
            if (p / "src" / "model" / "lstm.py").exists():
                return p
    return Path.cwd()


def resolve(p, root):
    """Relative paths: as given if they exist from the cwd, otherwise relative to the repo root."""
    if p is None:
        return None
    p = Path(p)
    if p.is_absolute() or p.exists():
        return str(p)
    return str(root / p)


def inspect(args):
    man = pd.read_csv(args.manifest)
    print("manifest:", args.manifest, "| split counts:", man.split.value_counts().to_dict())
    for f in FRACTIONS:
        p = Path(args.processed_root) / f"{f}pct"
        d = pd.read_parquet(p)
        print(f"\n[{f}%] {p} rows={len(d)} columns={list(d.columns)}")
        has_split = "split" in d.columns
        print("   `split` column in parquet:", has_split,
              "-> train.py used the OFFICIAL split" if has_split else
              "-> train.py fell back to a RANDOM 70/15/15 split (Ryuk would be in training!)")
        ov = d.rel_dir.isin(man.rel_dir).mean() if "rel_dir" in d.columns else 0
        print(f"   rel_dir found in manifest: {ov:.0%}")
        if "family" in d.columns:
            print("   Ryuk runs:", int((d.family == ZERODAY_FAMILY).sum()))
        if has_split:
            print("   parquet split counts:", d.split.value_counts().to_dict())
            if "family" in d.columns:
                print("   Ryuk per split:", d[d.family == ZERODAY_FAMILY].split.value_counts().to_dict())
        L = d.sequence.map(len)
        print(f"   seq_len min/mean/max = {L.min()}/{L.mean():.1f}/{L.max()}  | n_features = {len(d.sequence.iloc[0][0])}")
    print("\ncheckpoints:")
    for f in FRACTIONS:
        for c in sorted(glob.glob(os.path.join(args.ckpt_dir, f"lstm_attn_{f}pct_*.pt")), key=os.path.getmtime):
            ck = torch.load(c, map_location="cpu", weights_only=False)
            print(f"  {f}% {Path(c).name}  saved={pd.Timestamp(os.path.getmtime(c), unit='s'):%Y-%m-%d %H:%M} "
                  f"epoch={ck['epoch']} val_auc={ck['val_auc']:.4f} hidden={ck['args']['hidden_dim']}")


#  model / ckpt
def locate_checkpoints(args):
    """One checkpoint per cut. With MLflow: newest FINISHED run of the cut (and verify its split sizes).
    Without: newest .pt in --ckpt-dir for the cut."""
    found, warnings = {}, []
    runs = {}
    if args.tracking_uri:
        import mlflow
        from mlflow.tracking import MlflowClient
        mlflow.set_tracking_uri(args.tracking_uri)
        c = MlflowClient()
        exp = c.get_experiment_by_name(args.train_experiment)
        if exp is None:
            warnings.append(f"MLflow experiment '{args.train_experiment}' not found")
        else:
            for cut in FRACTIONS:
                r = c.search_runs([exp.experiment_id], f"tags.sequence_cut = '{cut}pct' and attributes.status = 'FINISHED'",
                                  order_by=["start_time DESC"], max_results=1)
                if r:
                    runs[cut] = r[0]
    for cut in FRACTIONS:
        pat = f"lstm_attn_{cut}pct_{runs[cut].info.run_id[:8]}.pt" if cut in runs else f"lstm_attn_{cut}pct_*.pt"
        cands = sorted(glob.glob(os.path.join(args.ckpt_dir, pat)), key=os.path.getmtime)
        if not cands:
            raise FileNotFoundError(f"no checkpoint for {cut}% matching {pat} in {args.ckpt_dir}")
        if args.select == "best_val" and len(cands) > 1:
            cands.sort(key=lambda c: torch.load(c, map_location="cpu", weights_only=False)["val_auc"])
        found[cut] = cands[-1]
        if len(cands) > 1:
            print(f"[{cut}%] {len(cands)} checkpoints match -> using {Path(cands[-1]).name} (--select {args.select})")
        if cut in runs:
            p = runs[cut].data.params
            found[f"{cut}_split"] = {k: int(p.get(k, -1)) for k in ("n_train", "n_val", "n_test")}
    return found, warnings


def build_model(ckpt, device):
    from src.model.lstm import RansomwareLSTM
    a = ckpt["args"]
    m = RansomwareLSTM(ckpt["n_features"], a["hidden_dim"], a["num_layers"], a["dropout"]).to(device)
    m.load_state_dict(ckpt["state_dict"])
    return m.eval()


@torch.no_grad()
def score(model, seqs, mean, std, device, bs=64):
    """Exactly train.py's path: log1p(clip) -> standardise with the TRAIN stats -> pad + true lengths."""
    out, peak = [], []
    for i in range(0, len(seqs), bs):
        chunk = seqs[i:i + bs]
        x = [torch.from_numpy(((np.log1p(np.clip(s, 0, None)) - mean) / std).astype(np.float32)) for s in chunk]
        lengths = torch.tensor([len(s) for s in x])
        s, w = model(pad_sequence(x, batch_first=True).to(device), lengths, return_attention=True)
        out += s.cpu().tolist()
        peak += (w.argmax(1).cpu().numpy() / lengths.numpy()).tolist()     
    return np.asarray(out), np.asarray(peak)


# metrics
def metrics(y, s, thr):
    pred = (s >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(n_pos=int(y.sum()), n_neg=int((1 - y).sum()), tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn),
                f1=f1_score(y, pred, zero_division=0), recall=tp / max(tp + fn, 1),
                precision=tp / max(tp + fp, 1), fpr=fp / max(fp + tn, 1),
                auc=roc_auc_score(y, s) if 0 < y.sum() < len(y) else np.nan)


def boot_ci(y, s, thr, n=1000, seed=0):
    rng, v = np.random.default_rng(seed), []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i):
            v.append(f1_score(y[i], (s[i] >= thr).astype(int), zero_division=0))
    return tuple(np.percentile(v, [2.5, 97.5])) if v else (np.nan, np.nan)


def choose_threshold(val, max_fpr=0.05):
    """Chosen on VAL only: centre of the plateau of best mean F1 over cuts, with mean FPR <= max_fpr."""
    sc = {}
    for t in np.round(np.arange(0.05, 0.96, 0.01), 2):
        ms = [metrics(y, s, t) for y, s in val.values()]
        if np.mean([m["fpr"] for m in ms]) <= max_fpr:
            sc[t] = np.mean([m["f1"] for m in ms])
    if not sc:
        return 0.5
    b = max(sc.values())
    return float(np.median([t for t, v in sc.items() if v >= b - 1e-9]))


#  main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--manifest", default=None, help="default: first existing of data/manifest.csv, manifest.csv, data/processed/manifest.csv")
    ap.add_argument("--processed-root", default="data/processed")
    ap.add_argument("--inspect", action="store_true")
    ap.add_argument("--merged-root", default=None)
    ap.add_argument("--fill-gaps", action="store_true", help="only with --merged-root; default matches the sequence notebook (no fill)")
    ap.add_argument("--ckpt-dir", default="checkpoints")
    ap.add_argument("--select", choices=["latest", "best_val"], default="latest", help="which checkpoint when several exist per cut")
    ap.add_argument("--repo-root", default=".", help="folder containing src/model/lstm.py")
    ap.add_argument("--tracking-uri", default=os.getenv("MLFLOW_TRACKING_URI"))
    ap.add_argument("--train-experiment", default="cipherstop-lstm-attention")
    ap.add_argument("--experiment", default="cipherstop-zeroday")
    ap.add_argument("--neg-split", default="test", help="benign negatives for zero-day (use 'train' only for a smoke test)")
    ap.add_argument("--out", default="zeroday_results")
    ap.add_argument("--no-mlflow-log", action="store_true")
    args = ap.parse_args()
    root = find_repo_root()
    print("repo root:", root)
    if args.manifest is None:
        cands = ["data/manifest.csv", "manifest.csv", "data/processed/manifest.csv", "data/raw/manifest.csv"]
        args.manifest = next((str(root / c) for c in cands if (root / c).exists()), None)
        if args.manifest is None:
            raise SystemExit(f"manifest.csv not found under {root}; pass --manifest PATH")
    args.manifest = resolve(args.manifest, root)
    args.processed_root = resolve(args.processed_root, root)
    args.merged_root = resolve(args.merged_root, root)
    args.ckpt_dir = resolve(args.ckpt_dir, root)
    args.out = str(root / args.out) if not Path(args.out).is_absolute() else args.out
    args.repo_root = str(root)
    if args.inspect:
        return inspect(args)
    sys.path.insert(0, os.path.abspath(args.repo_root))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)

    man = pd.read_csv(args.manifest)
    assert man[(man.family == ZERODAY_FAMILY) & man.split.isin(["train", "val", "test"])].empty, "Ryuk in train/val/test"
    expect = {"n_train": int((man.split == "train").sum()), "n_val": int((man.split == "val").sum()),
              "n_test": int((man.split == "test").sum())}

    ckpts, warns = locate_checkpoints(args)
    invalid = []
    for cut in FRACTIONS:
        got = ckpts.get(f"{cut}_split")
        if got and got != expect:
            invalid.append(f"{cut}%: MLflow n_train/val/test={got} but manifest={expect}")
    for w in warns + invalid:
        print("WARNING:", w)
    if invalid:
        print("!! The models were NOT trained on the manifest split (probably the random 70/15/15 fallback of "
              "train.py because the parquet had no `split` column). Ryuk may be in training: zero-day results INVALID. "
              "Retrain with a `split` column first.")

    rows, per = [], []
    zd_mask_names = ("zeroday", "zeroday_hw", "seen_test", "variants")
    val_cache, scores = {}, {}
    for f in FRACTIONS:
        ck = torch.load(ckpts[f], map_location=device, weights_only=False)
        model, mean, std = build_model(ck, device), ck["scaler_mean"], ck["scaler_std"]
        df = load_fraction(args, man, f)
        assert df.seq.iloc[0].shape[1] == ck["n_features"], "feature count differs from the checkpoint"
        assert not ((df.split.isin(["train", "val", "test"])) & (df.family == ZERODAY_FAMILY)).any()
        df["score"], df["attn_peak"] = score(model, list(df.seq), mean, std, device)
        df["seq_len"] = df.seq.map(len)
        scores[f] = df.drop(columns=["seq"])
        v = df[df.split == "val"]
        val_cache[f] = (v.label.values.astype(int), v.score.values)
        print(f"[{f}%] ckpt={Path(ckpts[f]).name} | runs={len(df)} | val AUC={roc_auc_score(*val_cache[f]):.3f}")

    thr_val = choose_threshold(val_cache)
    thrs = {"0.5": 0.5, "val": thr_val}
    print("thresholds:", thrs)

    for f, d in scores.items():
        neg = d[(d.split == args.neg_split) & (d.label == 0)]
        zd = d[d.family == ZERODAY_FAMILY]
        sets = {"zeroday": pd.concat([zd, neg]),
                "zeroday_hw": pd.concat([zd[zd.hw.isin(TEST_HW)], neg]),
                "seen_test": d[d.split == "test"],
                "variants": d[d.split == "variants"]}
        for name, e in sets.items():
            if e.empty or (name != "variants" and e.label.nunique() < 2):
                continue
            y, s = e.label.values.astype(int), e.score.values
            for tn_, t in thrs.items():
                m = metrics(y, s, t)
                lo, hi = boot_ci(y, s, t) if name != "variants" else (np.nan, np.nan)
                rows.append(dict(fraction=f, eval_set=name, thr_name=tn_, threshold=t, f1_ci_lo=lo, f1_ci_hi=hi, **m))
            per.append(e.assign(fraction=f, eval_set=name, pred_05=(s >= .5).astype(int), pred_val=(s >= thr_val).astype(int))
                       [["fraction", "eval_set", "rel_dir", "family", "hw", "split", "seq_len", "label", "score",
                         "pred_05", "pred_val", "attn_peak"]])

    S, R = pd.DataFrame(rows), pd.concat(per)
    S.to_csv(out / "results.csv", index=False)
    R.to_csv(out / "per_run_scores.csv", index=False)

    z = R[(R.eval_set == "zeroday") & (R.family == ZERODAY_FAMILY)]
    det = []
    for col in ("pred_05", "pred_val"):
        fl = z[z[col] == 1].groupby("rel_dir").fraction.min()
        v = pd.Series({r: fl.get(r, np.nan) for r in z.rel_dir.unique()})
        det.append(dict(threshold=col, n_runs=len(v), never_detected=int(v.isna().sum()), median_cut=v.median(),
                        p25=v.quantile(.25), p75=v.quantile(.75)))
    D = pd.DataFrame(det); D.to_csv(out / "detection_point.csv", index=False)

    ze = R[R.eval_set == "zeroday"]
    ze[(ze.label == 1) & (ze.pred_05 == 0)].to_csv(out / "false_negatives.csv", index=False)
    ze[(ze.label == 0) & (ze.pred_05 == 1)].to_csv(out / "false_positives.csv", index=False)

    piv = S[S.eval_set.isin(["zeroday", "seen_test"])].pivot_table(index=["thr_name", "fraction"], columns="eval_set", values="f1")
    if {"zeroday", "seen_test"} <= set(piv.columns):
        piv["gap_seen_minus_zeroday"] = piv["seen_test"] - piv["zeroday"]
    piv.to_csv(out / "gap_seen_vs_zeroday.csv")

    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for es, g in S[(S.thr_name == "0.5") & S.eval_set.isin(["zeroday", "seen_test"])].groupby("eval_set"):
        g = g.sort_values("fraction")
        for a_, c in zip(ax[:2], ["f1", "recall"]):
            a_.plot(g.fraction, g[c], marker="o", ls="-" if es == "zeroday" else "--", label=es)
    for a_, t in zip(ax[:2], ["F1", "Recall"]):
        a_.set_xticks(FRACTIONS); a_.set_xlabel("Prefix (%)"); a_.set_ylabel(t); a_.grid(alpha=.3); a_.legend()
    for f in FRACTIONS:
        zz = z[z.fraction == f]
        ax[2].hist(zz.score, bins=20, range=(0, 1), alpha=.4, label=f"{f}%")
    ax[2].axvline(.5, c="k", ls=":"); ax[2].set_title("Ryuk scores"); ax[2].legend()
    plt.tight_layout(); plt.savefig(out / "earliness.png", dpi=150)

    pd.set_option("display.width", 220)
    cols = ["fraction", "eval_set", "thr_name", "f1", "f1_ci_lo", "f1_ci_hi", "recall", "precision", "fpr", "auc", "n_pos", "n_neg"]
    print(S[S.eval_set != "variants"][cols].round(3).to_string(index=False))
    if (S.eval_set == "variants").any():
        print("\nvariants recall\n", S[S.eval_set == "variants"][["fraction", "thr_name", "recall"]].round(3).to_string(index=False))
    print("\ndetection point\n", D.to_string(index=False), "\n\ngap\n", piv.round(3).to_string())

    if args.tracking_uri and not args.no_mlflow_log:
        import mlflow
        mlflow.set_tracking_uri(args.tracking_uri); mlflow.set_experiment(args.experiment)
        with mlflow.start_run(run_name=f"zeroday-{ZERODAY_FAMILY}"):
            mlflow.set_tags({"eval": "zeroday", "heldout_family": ZERODAY_FAMILY,
                             "split_verified": str(not invalid), "scaler_source": "checkpoint"})
            mlflow.log_params({"val_threshold": thr_val, **{f"ckpt_{f}": Path(ckpts[f]).name for f in FRACTIONS}})
            for _, r in S[(S.eval_set == "zeroday") & (S.thr_name == "0.5")].iterrows():
                for k in ("f1", "recall", "precision", "fpr", "auc"):
                    mlflow.log_metric(f"zd_{int(r.fraction)}_{k}", float(r[k]))
            mlflow.log_artifacts(str(out))


if __name__ == "__main__":
    main()
''')
print('zeroday_eval_torch.py written')

zeroday_eval_torch.py written


In [ ]:
#  GPU + libraries 
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
try:
    import mlflow; print("mlflow", mlflow.__version__)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mlflow"], check=True)   # needs Internet ON
    import mlflow; print("mlflow", mlflow.__version__)

torch 2.11.0+cpu | cuda: False | device: CPU
mlflow 3.18.0


In [ ]:
#TRAIN 
def run(cmd):
    p = subprocess.Popen(cmd, cwd=ROOT, env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "agent.hint" in line or "FutureWarning" in line: continue
        print(line, end="")
    return p.wait()

rc = run([sys.executable, "train.py", "--data-root", PROC, "--ckpt-dir", CKPT, "--tracking-uri", TRACKING,
          "--experiment", EXPERIMENT, "--epochs", str(EPOCHS), "--patience", str(PATIENCE)])
assert rc == 0, "training failed - read the log above"

2026/11/23 14:48:57 INFO mlflow.tracking.fluent: Experiment with name 'cipherstop-lstm-attention' does not exist. Creating a new experiment.
2026/11/23 14:49:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/23 14:49:16 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/11/23 14:49:16 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requ

In [ ]:
# training results per cut (val/test, seen families) 
mlflow.set_tracking_uri(TRACKING)
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=["start_time DESC"])
cols = ["tags.sequence_cut", "params.n_train", "params.n_val", "params.n_test", "metrics.best_epoch",
        "metrics.final_val_auc", "metrics.final_val_f1", "metrics.final_test_auc", "metrics.final_test_f1", "metrics.final_test_fpr"]
res = runs[cols].drop_duplicates("tags.sequence_cut").sort_values("tags.sequence_cut")
print(res.to_string(index=False))
bad = res[res["metrics.final_val_auc"] < 0.9]
if len(bad): print("\nWARNING: val AUC < 0.9 for", list(bad["tags.sequence_cut"]), "- the model did not learn; fix training BEFORE trusting the zero-day result")

tags.sequence_cut params.n_train params.n_val params.n_test  metrics.best_epoch  metrics.final_val_auc  metrics.final_val_f1  metrics.final_test_auc  metrics.final_test_f1  metrics.final_test_fpr
            10pct            770          220           330                 8.0               0.974583              0.901961                0.980074               0.912621                0.100000
            25pct            770          220           330                 6.0               0.999917              0.969072                0.999704               0.969283                0.005556
            50pct            770          220           330                 6.0               1.000000              1.000000                0.999926               0.993289                0.000000
            75pct            770          220           330                 7.0               1.000000              0.995025                1.000000               1.000000                0.000000


In [ ]:
#  ZERO-DAY test (Ryuk never seen) 
rc = run([sys.executable, "zeroday_eval_torch.py", "--manifest", MANIFEST, "--processed-root", PROC, "--ckpt-dir", CKPT,
          "--repo-root", ROOT, "--out", OUT, "--tracking-uri", TRACKING, "--train-experiment", EXPERIMENT])
assert rc == 0, "zero-day evaluation failed - read the log above"

2026/11/23 14:57:17 INFO mlflow.tracking.fluent: Experiment with name 'cipherstop-zeroday' does not exist. Creating a new experiment.
repo root: /kaggle/working/CipherStop
[10%] ckpt=lstm_attn_10pct_23f9cd43.pt | runs=1470 | val AUC=0.975
[25%] ckpt=lstm_attn_25pct_0ae8e8dc.pt | runs=1470 | val AUC=1.000
[50%] ckpt=lstm_attn_50pct_2331ba5a.pt | runs=1470 | val AUC=1.000
[75%] ckpt=lstm_attn_75pct_c81372d3.pt | runs=1470 | val AUC=1.000
thresholds: {'0.5': 0.5, 'val': 0.195}
 fraction   eval_set thr_name    f1  f1_ci_lo  f1_ci_hi  recall  precision   fpr   auc  n_pos  n_neg
       10    zeroday      0.5 0.703     0.636     0.767   0.607      0.835 0.100 0.888    150    180
       10    zeroday      val 0.784     0.731     0.832   0.760      0.809 0.150 0.888    150    180
       10 zeroday_hw      0.5 0.609     0.459     0.730   0.700      0.538 0.100 0.897     30    180
       10 zeroday_hw      val 0.593     0.456     0.705   0.800      0.471 0.150 0.897     30    180
       10  seen_

In [ ]:
# package everything to download 
stage = f"{WORK}/cipherstop_run"
shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage)
for src, name in ((CKPT, "checkpoints"), (OUT, "zeroday_results"), (MLRUNS, "mlruns")):
    if os.path.exists(src): shutil.copytree(src, f"{stage}/{name}")
shutil.make_archive(f"{WORK}/cipherstop_run", "zip", stage)
print("zip:", f"{WORK}/cipherstop_run.zip", f"({os.path.getsize(WORK + '/cipherstop_run.zip') / 1e6:.0f} MB)")

zip: /kaggle/working/cipherstop_run.zip (10 MB)
